# Lesson 12: Hardware-Aware Optimization — ONNX Graph Passes, OpenVINO NNCF, Apple MLX & Qualcomm QNN

In this companion notebook, you will:
1. Export a PyTorch model to **ONNX** and apply **ONNX Runtime dynamic INT8 quantization** (`quantize_dynamic`).
2. Compress weights with **Intel OpenVINO NNCF** (`nncf.compress_weights`) and benchmark inference latency.
3. Run an automated hardware-target framework selector across x86, Apple Silicon, Snapdragon NPU, and cross-platform CPU.

In [ ]:
!pip install -q -U torch onnx onnxruntime openvino nncf

## 1. ONNX Dynamic INT8 Quantization Benchmark
We build a linear-stack PyTorch block, export to ONNX, quantize to INT8 with `onnxruntime.quantization`, and compare file sizes and latency.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

class MiniFFN(nn.Module):
    def __init__(self, dim=1024):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))
    def forward(self, x):
        return self.net(x)

model = MiniFFN().eval()
dummy = torch.randn(1, 32, 1024)
torch.onnx.export(model, dummy, "ffn_fp32.onnx", input_names=["x"], output_names=["y"], opset_version=14)
quantize_dynamic("ffn_fp32.onnx", "ffn_int8.onnx", weight_type=QuantType.QUInt8)

fp32_mb = os.path.getsize("ffn_fp32.onnx") / (1024 ** 2)
int8_mb = os.path.getsize("ffn_int8.onnx") / (1024 ** 2)
print(f"ONNX FP32: {fp32_mb:.2f} MB -> ONNX INT8: {int8_mb:.2f} MB ({(1 - int8_mb/fp32_mb)*100:.1f}% smaller)")

## 2. Intel OpenVINO IR Conversion & Inference
Convert the PyTorch module directly to OpenVINO IR and run compiled inference on CPU.

In [ ]:
import openvino as ov

core = ov.Core()
ov_model = ov.convert_model(model, example_input=dummy)
compiled = core.compile_model(ov_model, "CPU")

x_np = dummy.numpy()
t0 = time.perf_counter()
for _ in range(20):
    _ = compiled({0: x_np})
avg_ms = ((time.perf_counter() - t0) / 20) * 1000.0
print(f"OpenVINO CPU average latency: {avg_ms:.2f} ms")